# 05 — Drift Detection: Version Lineage as a Time Axis

**Tier 2 — Boundaries & robustness** · [GenAI Alignment scenario library](../README.md#scenario-library) · native — no sibling repo tests this

> **In one sentence:** does the same simulated system stay behaviorally stable as the model underneath it changes, and would this harness actually notice if it didn't?

| | |
|---|---|
| **Risk if untested** | Outputs change over time with no input change, driven by silent model, tool, or prompt updates. |
| **What this tests** | Behavior is stable absent input change; any material change is detected, explained, and gated. |

**A single notebook run happens at one point in time — "drift" is a question about behavior *over* time.** This scenario resolves that by using **model version as the time axis**: vendor-shipped dated snapshots of the same model family are themselves real, time-separated data points, available without waiting for calendar time to pass. `DRIFT_MODEL_SEQUENCE` (`.env`) holds that lineage; `DRIFT_FLOATING_MODEL` adds one more comparison against a live, undated, auto-updating alias — a cheap stand-in for the *other* kind of drift (a floating deployment silently changing behavior with no version bump to see).

Three things measured here, all against the exact same HR/IT golden set [Intended Performance](../docs/intended_performance.md) already scores for correctness — no new dataset authored for this scenario:

1. **Noise floor per version** — N repeats at every snapshot, so a real version-to-version difference and ordinary run-to-run stochastic noise aren't confused for each other.
2. **Cross-version drift scoring** — does a later version's output still match the baseline's own dominant answer, on both a score axis and a semantic axis, "material" only when the shift exceeds what the baseline's own noise floor would explain.
3. **Harness validation** — a drift detector that's never been shown to detect anything, or to stay quiet on nothing, isn't validated (see [`docs/drift_detection.md`](../docs/drift_detection.md#one-month-isnt-enough-to-see-drift-test-the-control-not-the-calendar)). Two synthetic controls against the same baseline snapshot confirm the pipeline actually has the power to do both.

**A real finding from building this notebook, not a hypothetical:** 4 of 6 originally candidate dated snapshots had already been retired by this deployment before this scenario could test against them — confirmed via live calls returning HTTP 410, not assumed. See Methodology below.

This notebook is code-light — everything above lives in [`scenarios/drift_detection.py`](../scenarios/drift_detection.py).

## ⚙️ Setup

```bash
pip install -e .
cp .env.example .env   # then fill in DRIFT_MODEL_SEQUENCE (and optionally DRIFT_FLOATING_MODEL)
```

`DRIFT_MODEL_SEQUENCE` needs at least 2 dated snapshots from the same model family — see `.env.example` for the exact format. New here? See [README — Setup](../README.md#setup) first. The cell below checks what's actually present in *this* kernel and stops cleanly if anything's missing, rather than failing deep in a later cell after API calls have already started.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Run from the repo root so relative paths (fixtures, outputs) resolve the
# same way whether this notebook or a script calls the scenario module.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
load_dotenv(Path.cwd() / ".env")

from scenarios import drift_detection as scenario
from scenarios import intended_performance as ip_scenario
from reporting.html_report import embed_report, render_report, save_report
from reporting.env_check import check_environment
from reporting.artifacts import artifact_trail
from reporting.display import GENERIC_JUDGE_MODEL_NAME, GENERIC_PROVIDER_NAME

pd.set_option("display.max_colwidth", 160)

### Environment Check

In [ ]:
ready = check_environment(
    required_packages=["genai_capability_bench", "jinja2", "matplotlib"],
    required_env_vars=["OPENAI_API_KEY", "OPENAI_BASE_URL", "OPENAI_API_VERSION", "DRIFT_MODEL_SEQUENCE", "JUDGE_MODEL"],
)
assert ready, "Fix the items above before continuing — later cells will spend real API calls."

<a id="methodology"></a>
## 📐 Methodology

**The target system, shared with Intended Performance.** The exact same `RAG_SYSTEM_PROMPT` mandate plus per-question knowledge-base document, run against every version in the lineage — only the model deployment changes, one axis at a time.

**Why 2 live snapshots instead of the 6 originally planned.** The candidate lineage started as 6 dated GPT-5.x snapshots spanning ~8 months. A pre-build smoke test against every candidate found that 4 of them — the 4 oldest — now return HTTP 410 ("deployment retired") on this Azure deployment. That's not a workaround-able bug; it's the deployment itself no longer serving those snapshots. Rather than discard that as an inconvenience, it's kept as a documented finding: **a version lineage this scenario has to plan around a vendor retiring older pinned snapshots is itself evidence for the risk this scenario tests.** See [Limitations & Future Work](../docs/drift_detection.md#limitations--future-work).

**Three tracks:**

1. **Version sweep** — `scenario.N_REPEATS` repeats of the golden set against each live dated snapshot in `DRIFT_MODEL_SEQUENCE`, oldest first. `scenario.version_noise_floor` computes each version's own internal consistency (reusing `reporting/repeat_run.py`'s existing variance + bidirectional-entailment machinery, unchanged from Consistency & Reliability's use of the same functions).
2. **Cross-version drift scoring** — `scenario.score_drift_vs_baseline` compares each candidate version's answers against the earliest version's dominant answer, on a deterministic-score axis and a semantic-entailment axis. A shift only counts as **material** when it falls outside a Wilson confidence interval built from the candidate's own repeats and centered on the baseline's point estimate — the same tolerance-band-not-eyeballed principle this repo's reliability-significance test already applies elsewhere, not a raw diff against an arbitrary cutoff.
3. **Harness validation** — `scenario.run_control_validation` runs two synthetic batches against the *same* baseline snapshot: an unperturbed second batch (expect: no material drift — the false-positive check) and a batch run against a system prompt deliberately corrupted to double every cited policy number (expect: material drift on most tasks — the detection-power check). Both use the real drift-scoring pipeline above, not a separate stub.

**Floating-alias check (optional, if `DRIFT_FLOATING_MODEL` is set):** the same drift-scoring pipeline again, comparing a live, undated, auto-updating alias against the most recent *pinned* snapshot in the sweep — the closest this scenario gets to the silent/calendar-drift risk without literally waiting for real time to pass.

## 🗂️ Data

Reused, not new — the exact same 10-question HR/IT policy golden set [Intended Performance](../docs/intended_performance.md) scores for correctness, loaded directly from `scenarios.intended_performance`.

In [ ]:
golden_set = ip_scenario.load_golden_set()
print(f"HR/IT policy golden set — {len(golden_set)} tasks (reused from Intended Performance)")
golden_set[["task_id", "subcategory", "trap_type", "expected_output"]]

## 🕰️ Version Lineage

The actual sequence this run tests, pulled live from `.env` — dates are real (not confidential); only the underlying deployment name is masked, see [`reporting/display.py`](../reporting/display.py).

In [ ]:
sequence = scenario.load_version_sequence()
floating_entry = scenario.load_floating_entry(sequence)

rows = [{"label": e["label"], "date": e["date"].isoformat()} for e in sequence]
if floating_entry:
    rows.append({"label": floating_entry["label"], "date": "live / undated"})
pd.DataFrame(rows)

## ▶️ Run — Version Sweep

**What this cell actually spends:** `N_REPEATS × len(golden_set) × len(sequence)` target API calls, plus a variable number of judge calls for the internal-consistency check on top.

In [ ]:
judge_client = scenario.build_judge_client(sequence[0]["deployment"])
display(Markdown(f"**LLM Provider:** {GENERIC_PROVIDER_NAME}  \n**Judge model:** `{GENERIC_JUDGE_MODEL_NAME}`"))

sweep_results = scenario.run_version_sweep(golden_set, sequence, n=scenario.N_REPEATS)
noise_floor = scenario.version_noise_floor(sweep_results, judge_client)
print(f"{len(sweep_results)} rows collected ({scenario.N_REPEATS} repeats \u00d7 {len(golden_set)} tasks \u00d7 {len(sequence)} version(s))")
noise_floor[["version_label", "task_id", "avg_score", "score_std", "pass_rate", "semantic_consistency"]]

In [ ]:
trajectory_chart = scenario.plot_trajectory(noise_floor)

## 📊 Cross-Version Drift Scoring

Every candidate version scored against the earliest (baseline) version — `material_drift` is `True` only when the shift on either axis falls outside what the baseline's own noise floor would explain.

In [ ]:
sweep_drift = scenario.build_sweep_drift_table(sweep_results, noise_floor, judge_client)
sweep_drift

## 🔄 Floating-Alias Check

Skipped automatically if `DRIFT_FLOATING_MODEL` isn't set in `.env`.

In [ ]:
if floating_entry:
    floating_results = scenario.run_floating_check(golden_set, floating_entry, n=scenario.N_REPEATS)
    reference_label = sequence[-1]["label"]
    reference_results = sweep_results[sweep_results["version_label"] == reference_label]
    reference_var = noise_floor[noise_floor["version_label"] == reference_label]
    floating_drift = scenario.build_reference_drift_table(
        floating_results, reference_results, reference_var, judge_client, floating_entry["label"],
    )
    floating_chart = scenario.plot_drift_by_task(
        floating_drift,
        "Floating alias vs. last pinned snapshot",
        "Live, undated auto-updating alias compared against the most recent pinned version in the "
        "lineage \u2014 the closest this run gets to silent/calendar drift without waiting for real time to pass.",
    )
else:
    floating_results, floating_drift, floating_chart = None, None, None
    print("DRIFT_FLOATING_MODEL not set \u2014 skipping the floating-alias check.")
floating_drift

## 🧪 Harness Validation — Test the Control, Not the Calendar

Two synthetic batches against the *same* baseline snapshot, run through the identical drift-scoring pipeline used above: an unperturbed second batch (expect quiet) and a batch with a deliberately corrupted system prompt that doubles every cited policy number (expect most tasks flagged). If either control doesn't land where expected, that's a caveat on trusting the version-sweep's own `material_drift` flags this run, not just a footnote — see Key Findings in the report below.

In [ ]:
control_results = scenario.run_control_validation(golden_set, sequence[0]["deployment"], n=scenario.N_REPEATS)

baseline_label = sequence[0]["label"]
baseline_results = sweep_results[sweep_results["version_label"] == baseline_label]
baseline_var = noise_floor[noise_floor["version_label"] == baseline_label]

unchanged_candidates = control_results[control_results["version_label"] == "control: unchanged (2nd batch)"]
corrupted_candidates = control_results[control_results["version_label"] == "control: injected corruption"]

unchanged_drift = scenario.build_reference_drift_table(
    unchanged_candidates, baseline_results, baseline_var, judge_client, "control: unchanged (2nd batch)",
)
corrupted_drift = scenario.build_reference_drift_table(
    corrupted_candidates, baseline_results, baseline_var, judge_client, "control: injected corruption",
)
control_chart = scenario.plot_control_validation(unchanged_drift, corrupted_drift)

<a id="reporting-template"></a>
## 📝 Testing Report

Built from this run's data through the same [uniform HTML template](../reporting/templates/scenario_report.html.j2) every scenario in this repo uses: **Executive Summary → Key Findings → Testing Scope → Testing Approach → Results Summary → High-Risk Cases (if any) → Next Steps → Appendix.** The Appendix carries the noise-floor table, every results table above, and a live-checked artifact trail.

In [ ]:
saved_paths = scenario.save_artifacts(
    sweep_results, noise_floor, sweep_drift, floating_results, floating_drift,
    control_results, unchanged_drift, corrupted_drift,
)
artifacts_table = artifact_trail(scenario.artifacts(saved_paths))

charts = [c for c in [trajectory_chart, floating_chart, control_chart] if c is not None]
report = scenario.build_report(
    sequence, floating_entry, noise_floor, sweep_drift, floating_drift,
    unchanged_drift, corrupted_drift, charts, artifacts_table,
)

html = render_report(report)
report_path = save_report(html, "outputs/reports/drift_detection.html")
print(f"Report saved to {report_path}")
embed_report(html)

<a id="how-to-extend"></a>
## 🔧 How to Extend This Scenario

- **Widen the lineage** — `DRIFT_MODEL_SEQUENCE` (`.env`) takes any number of `ISO_DATE:deployment` pairs \u2265 2; a longer-lived or explicitly version-pinned deployment tier would let this run a richer trajectory than the 2 points that survived retirement this time.
- **Run it on a cadence** — this notebook demonstrates the mechanism works once; the design doc's actual "per release / set cadence" repeat loop (`docs/drift_detection.md`) needs this wired to run automatically, not just on demand.
- **Stress-test the material-drift threshold** — `scenario.INJECTED_DRIFT_SYSTEM_SUFFIX` is a deliberately blunt corruption (doubles every number); a subtler perturbation would test whether the Wilson-CI threshold is sensitive enough to catch a more realistic drift event, not just an obvious one.
- **Add a dedicated NLI model for the entailment check** — same open item as Consistency & Reliability: `bidirectional_entailment` uses an LLM-judge prompt rather than a dedicated NLI model, which removes one source of judge-model variance from the semantic-drift signal at the cost of a new ML dependency.
- **Different target system** — everything above is wired to Intended Performance's golden set and RAG mandate; a different simulated system would need `run_version_sweep`'s `_run_once` closure pointed at a different scenario module's run/score functions instead.